# Notebook 4: Environmental Risk

Adds two environmental risk modifiers:
- **Contamination proximity**: distance to nearest EPA Toxics Release Inventory (TRI) facility in WA
- **Waterway sensitivity**: proximity to major regulated rivers (Columbia, Snake, Spokane, Skagit, Yakima)
  as a proxy for ESA thermal discharge risk and water withdrawal scrutiny.

| Layer | Source |
|---|---|
| TRI industrial facilities | EPA TRI Envirofacts REST API (`TRI_FACILITY/STATE_ABBR/WA`) |
| Major waterways | Sample points along key WA rivers, IDW to grid |
| Grid | data/WA/grid_scores.geojson (NB3 output, 974 cells) |

**Note on data source change (2026-06-11):** The EPA Envirofacts `SEMS_ACTIVE_SITES` table (NPL Superfund sites)
was removed from the API. TRI replaces it: 556 geocoded WA industrial facilities vs 26 hand-entered NPL sites.
TRI is a broader but more comprehensive contamination indicator — it covers all facilities releasing toxic
chemicals, not just the worst ~1% that reach NPL status.

In [1]:
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

warnings.filterwarnings("ignore")

RAW       = Path("../data/raw")
PROCESSED = Path("../data/processed")
DARK_BG   = "#1a1a2e"
WHITE     = "white"

plt.rcParams.update({
    "text.color": WHITE, "axes.labelcolor": WHITE,
    "xtick.color": WHITE, "ytick.color": WHITE, "font.size": 16,
})
print("Imports OK")


/home/simonhans/anaconda3/lib/python3.7/site-packages/geopandas/_compat.py:115: UserWarning: The Shapely GEOS version (3.11.4-CAPI-1.17.4) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  shapely_geos_version, geos_capi_version_string


Imports OK


## 1. Load grid and boundaries


In [2]:
wa     = gpd.read_file(RAW / "wa_state.geojson").to_crs("EPSG:32610")
dc_gdf = gpd.read_file(RAW / "datacenters.geojson").to_crs("EPSG:32610")
grid   = gpd.read_file("../static/grid_scores.geojson").to_crs("EPSG:32610")

grid["centroid"] = grid.geometry.centroid
tgt_x = np.array([c.x for c in grid.centroid])
tgt_y = np.array([c.y for c in grid.centroid])
grid_coords = np.column_stack([tgt_x, tgt_y])

print(f"Grid: {len(grid)} cells")
print(f"Columns: {list(grid.columns)}")


Grid: 974 cells
Columns: ['cell_id', 'tx_score', 'water_score', 'ej_score', 'seismic_score', 'flood_score', 'contamination_score', 'waterway_score', 'geometry', 'centroid']


## 2. Contamination Proximity

EPA Toxics Release Inventory (TRI) facilities from the
[Envirofacts REST API](https://data.epa.gov/efservice/TRI_FACILITY/STATE_ABBR/WA/JSON).
Covers all WA facilities that report toxic chemical releases — a broader proxy for
industrial contamination risk than NPL Superfund sites alone.

TRI stores longitude as a positive value; negated here to get West-hemisphere decimal degrees.

In [ ]:
import requests

tri_path = RAW / "tri_facilities_wa.csv"

if not tri_path.exists():
    print("Fetching EPA TRI facilities for WA...")
    url = "https://data.epa.gov/efservice/TRI_FACILITY/STATE_ABBR/WA/JSON"
    r = requests.get(url, timeout=60, headers={"User-Agent": "datacenter-siting-research/1.0"})
    r.raise_for_status()
    data = r.json()
    df_tri = pd.DataFrame(data)
    df_tri["lat"] = pd.to_numeric(df_tri["pref_latitude"], errors="coerce")
    df_tri["lon"] = -pd.to_numeric(df_tri["pref_longitude"], errors="coerce")  # TRI stores as positive
    df_tri["name"] = df_tri["facility_name"]
    df_tri = df_tri[["lat", "lon", "name"]].dropna(subset=["lat", "lon"])
    df_tri.to_csv(tri_path, index=False)
    print(f"Saved {len(df_tri)} TRI facilities to cache.")
else:
    df_tri = pd.read_csv(tri_path)
    print(f"Loaded {len(df_tri)} TRI facilities from cache.")

tri_gdf = gpd.GeoDataFrame(
    df_tri,
    geometry=gpd.points_from_xy(df_tri["lon"], df_tri["lat"]),
    crs="EPSG:4326"
).to_crs("EPSG:32610")

tri_coords = np.column_stack([tri_gdf.geometry.x, tri_gdf.geometry.y])
tree_tri = cKDTree(tri_coords)
dist_tri, _ = tree_tri.query(grid_coords, k=1)

grid["contamination_score"] = dist_tri / dist_tri.max()
print(f"contamination_score: {grid.contamination_score.min():.3f} - {grid.contamination_score.max():.3f}")

## 3. Waterway Sensitivity

Sample points along the Columbia, Snake, Spokane, Skagit, and Yakima rivers.
Proximity is a proxy for ESA Section 7 thermal discharge constraints
and water withdrawal regulatory scrutiny.
Closer = higher regulatory risk = lower score.


In [4]:
# Sample points along major WA regulated waterways
columbia_pts = [
    (48.98,-117.63),(48.60,-118.10),(48.30,-118.45),(47.95,-118.85),
    (47.75,-119.15),(47.55,-119.45),(47.30,-119.65),(47.10,-119.65),
    (46.70,-119.60),(46.45,-119.40),(46.30,-119.15),(46.20,-119.00),
    (46.10,-118.97),(45.95,-119.48),(45.85,-119.85),(45.75,-120.40),
    (45.70,-121.00),(45.65,-121.60),(45.62,-122.20),(45.60,-122.75),
]
snake_pts = [
    (46.40,-117.05),(46.35,-117.45),(46.30,-118.00),
    (46.28,-118.50),(46.24,-119.05),
]
spokane_pts = [
    (47.66,-117.42),(47.68,-117.80),(47.72,-118.10),(47.78,-118.40),
]
skagit_pts = [
    (48.72,-121.20),(48.65,-121.60),(48.55,-121.90),
    (48.45,-122.10),(48.40,-122.25),(48.47,-122.40),
]
yakima_pts = [
    (46.61,-120.51),(46.42,-120.12),(46.28,-119.87),(46.22,-119.62),
]

all_pts = columbia_pts + snake_pts + spokane_pts + skagit_pts + yakima_pts
riv_df  = pd.DataFrame(all_pts, columns=["lat","lon"])
riv_gdf = gpd.GeoDataFrame(
    riv_df,
    geometry=gpd.points_from_xy(riv_df.lon, riv_df.lat),
    crs="EPSG:4326"
).to_crs("EPSG:32610")

riv_coords = np.column_stack([riv_gdf.geometry.x, riv_gdf.geometry.y])
tree_riv = cKDTree(riv_coords)
dist_riv, _ = tree_riv.query(grid_coords, k=1)

grid["waterway_score"] = dist_riv / dist_riv.max()
print(f"waterway_score: {grid.waterway_score.min():.3f} - {grid.waterway_score.max():.3f}")


waterway_score: 0.009 - 1.000


## 4. Maps


In [ ]:
layers = [
    ("contamination_score", "Contamination Proximity",
     "(1 = far from TRI industrial facility)"),
    ("waterway_score",       "Waterway Sensitivity",
     "(1 = far from major regulated river)"),
]

fig, axes = plt.subplots(1, 2, figsize=(20, 9), facecolor=DARK_BG)

for ax, (col, title, subtitle) in zip(axes, layers):
    ax.set_facecolor(DARK_BG)
    wa.boundary.plot(ax=ax, color="#4a4a6a", linewidth=1.0, zorder=1)
    n_before = len(fig.axes)
    grid.plot(column=col, ax=ax, cmap="RdYlGn", vmin=0, vmax=1,
              legend=True,
              legend_kwds={"shrink": 0.65, "label": "0=poor / 1=ideal"},
              alpha=0.85, zorder=2)
    if len(fig.axes) > n_before:
        cb = fig.axes[-1]
        cb.tick_params(labelsize=14, colors=WHITE)
        cb.yaxis.label.set_color(WHITE)
        cb.yaxis.label.set_size(14)
    _rep  = dc_gdf[dc_gdf["source"] == "reported"]
    _prop = dc_gdf[dc_gdf["source"] == "proposed"]
    ax.scatter(_rep.geometry.x,  _rep.geometry.y,
               c=WHITE, s=120, marker="D", zorder=5,
               edgecolors="black", linewidths=0.8)
    ax.scatter(_prop.geometry.x, _prop.geometry.y,
               facecolors="none", s=120, marker="D", zorder=5,
               edgecolors="black", linewidths=1.5)
    ax.set_title(f"{title}\n{subtitle}", color=WHITE, fontsize=20,
                 pad=10, linespacing=1.4)
    ax.set_xlabel(""); ax.set_ylabel("")
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    for spine in ax.spines.values():
        spine.set_edgecolor("#4a4a6a")

plt.suptitle(
    "Washington State: Environmental Risk Modifiers\n"
    "White filled = existing DC  /  outline = proposed DC",
    color=WHITE, fontsize=22, y=0.90
)
plt.tight_layout(rect=[0, 0, 1, 0.86])
plt.savefig(PROCESSED / "environmental_risk.png", dpi=150,
            bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print("Saved to data/processed/environmental_risk.png")

## 5. Export updated grid_scores.geojson


In [6]:
grid_out = grid.drop(columns=["centroid"]).to_crs("EPSG:4326")
grid_out.to_file("../static/grid_scores.geojson", driver="GeoJSON")
print(f"Saved ../static/grid_scores.geojson")
print(f"Columns: {list(grid_out.columns)}")


Saved ../static/grid_scores.geojson
Columns: ['cell_id', 'tx_score', 'water_score', 'ej_score', 'seismic_score', 'flood_score', 'contamination_score', 'waterway_score', 'geometry']


## 6. Key Findings


In [7]:
print("=== Proposed Sites ===")
for _, row in dc_gdf[dc_gdf["source"] == "proposed"].iterrows():
    pt = row.geometry
    dists = grid.centroid.apply(lambda c: pt.distance(c))
    n = grid.loc[dists.idxmin()]
    print(f'  {row["name"]}:\n'
          f'    contamination={n.contamination_score:.3f}'
          f'  waterway={n.waterway_score:.3f}\n')

print("=== Existing Clusters ===")
seen = set()
for _, row in dc_gdf[dc_gdf["source"] == "reported"].iterrows():
    pt = row.geometry
    dists = grid.centroid.apply(lambda c: pt.distance(c))
    n = grid.loc[dists.idxmin()]
    key = (round(n.contamination_score,2), round(n.waterway_score,2))
    if key not in seen:
        seen.add(key)
        print(f'  {row["name"]}:\n'
              f'    contamination={n.contamination_score:.3f}'
              f'  waterway={n.waterway_score:.3f}\n')


=== Proposed Sites ===
  Digital Realty (proposed):
    contamination=0.009  waterway=0.463

  Amazon Wallula Gap (proposed):
    contamination=0.014  waterway=0.015

  Atlas Agro Richland DC1 (proposed):
    contamination=0.142  waterway=0.046

  Trammell Crow Lewis Clark (proposed):
    contamination=0.207  waterway=0.037

=== Existing Clusters ===
  Microsoft Quincy Campus:
    contamination=0.501  waterway=0.068

  Microsoft EAT06/EAT09:
    contamination=0.554  waterway=0.244

  Equinix SE2 Seattle:
    contamination=0.009  waterway=0.463

  HorizonIQ Seattle (Tukwila):
    contamination=0.028  waterway=0.544

  Verizon Liberty Lake:
    contamination=0.145  waterway=0.116

